# SemEval 2026 EmoVA — Final Evaluation (Task 1)

**Flow:**
1. Train on the **full** training set for exactly `N` epochs (from `config.json`)
2. Predict on the test set
3. Evaluate against official gold labels using the official metrics

> Set `CONFIG_PATH` in *Cell 5* to point to the `config.json` produced by your best ablation run.

In [ ]:
import os

def setup_storage():
    # Google Colab
    if "COLAB_GPU" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive")
        base_dir = "/content/drive/MyDrive"
        env = "colab"
    # Kaggle
    elif os.path.exists("/kaggle"):
        base_dir = "/kaggle/working"
        env = "kaggle"
    # Local fallback
    else:
        base_dir = os.getcwd()
        env = "local"

    print(f"Running on : {env}")
    print(f"Base dir   : {base_dir}")
    return base_dir, env


BASE_DIR, ENV = setup_storage()

In [ ]:
PROJECT_ROOT = f"{BASE_DIR}/SEMEVAL2026_EMOVA"
CKPT_DIR     = f"{PROJECT_ROOT}/model_checkpoints_final"

os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
!git clone https://github.com/AndreaLolli2912/SemEval2026-EmoVA.git
%cd SemEval2026-EmoVA

In [ ]:
import json
import random
from datetime import datetime
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader

from src.data.dataset import EmoVADataset
from src.data.collate import create_collate_fn
from src.models.affect_model import AffectModel
from src.models.tokenizer_wrapper import TokenizerWrapper
from src.training import train_epoch, GradientClipper
from src.evaluation.metrics import (
    evaluate_subtask1,
    collect_predictions_for_eval,
    print_evaluation_results,
)

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    torch.use_deterministic_algorithms(True)

## Config

Point `CONFIG_PATH` to the `config.json` saved by your best ablation run.
All string-encoded values (`"True"`, `"32"`, …) are automatically converted back to proper Python types.

In [ ]:
# ── EDIT THIS PATH ────────────────────────────────────────────────────────────
CONFIG_PATH = f"{PROJECT_ROOT}/model_checkpoints/<YOUR_RUN_FOLDER>/config.json"
# ──────────────────────────────────────────────────────────────────────────────


def _parse(v):
    """Convert string-encoded values produced by the trainer back to Python types."""
    if not isinstance(v, str):
        return v
    if v == 'True':  return True
    if v == 'False': return False
    if v == 'None':  return None
    try:    return int(v)
    except ValueError: pass
    try:    return float(v)
    except ValueError: pass
    return v


with open(CONFIG_PATH) as f:
    raw = json.load(f)

cfg_dict = {k: _parse(v) for k, v in raw.items()}
config   = type('Config', (), cfg_dict)()

# ── Derived paths (override data_path to local project structure) ─────────────
config.data_path = f"{PROJECT_ROOT}/dataset/train_subtask1.csv"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(config.seed)

print(f"Model      : {config.model_name}")
print(f"Epochs     : {config.epochs}")
print(f"Loss       : {config.loss}")
print(f"LoRA       : {getattr(config, 'encoder_lora', False)}")
print(f"BitFit     : {getattr(config, 'encoder_bitfit', False)}")
print(f"Device     : {device}")

## Data — Full Training Set (no validation split)

In [ ]:
tokenizer    = TokenizerWrapper(config.model_name, config.max_text_length)
full_dataset = EmoVADataset(
    path=config.data_path,
    dtype=torch.float32,
    constrain_output=config.constrain_output,
)
collate_fn   = create_collate_fn(tokenizer)

train_loader = DataLoader(
    full_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=config.num_workers,
)

print(f"Full training dataset: {len(full_dataset)} users")

## Model

In [ ]:
model = AffectModel(
    model_path=config.model_name,
    encoder_bitfit=getattr(config, 'encoder_bitfit', False),
    encoder_use_lora=getattr(config, 'encoder_lora', False),
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    n_heads=config.n_heads,
    isab_inducing_points=config.isab_inducing_points,
    pma_num_seeds=config.pma_num_seeds,
    lstm_hidden_dim=config.lstm_hidden_dim,
    lstm_num_layers=config.lstm_num_layers,
    lstm_bidirectional=config.lstm_bidirectional,
    dropout=config.dropout,
    constrain_output=config.constrain_output,
)

if getattr(config, 'encoder_bitfit', False) or getattr(config, 'encoder_lora', False):
    model.encoder.backbone.gradient_checkpointing_enable()

model = model.to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {total:,} total, {trainable:,} trainable")

## Optimizer / Scheduler / Clipper

In [ ]:
param_groups = [
    {'params': [p for n, p in model.encoder.named_parameters() if p.requires_grad],
     'lr': 5e-6, 'name': 'encoder'},
    ({'params': list(model.isab.parameters()), 'lr': config.lr, 'name': 'isab'}
     if model.isab else None),
    ({'params': list(model.pma.parameters()), 'lr': config.lr, 'name': 'pma'}
     if model.pma else None),
    {'params': list(model.lstm.parameters()), 'lr': config.lr, 'name': 'lstm'},
    {'params': list(model.head.parameters()), 'lr': config.lr, 'name': 'head'},
]
param_groups = [pg for pg in param_groups if pg is not None and len(pg['params']) > 0]

optimizer = AdamW(param_groups, weight_decay=config.weight_decay)
# Step scheduler on train loss (no val set available)
scheduler = ReduceLROnPlateau(
    optimizer, mode='min',
    factor=config.scheduler_factor,
    patience=config.scheduler_patience,
)
clipper   = GradientClipper(max_norm=config.max_grad_norm)

for pg in optimizer.param_groups:
    n = sum(p.numel() for p in pg['params'])
    print(f"{pg['name']:12s}: {n:>12,} params  lr={pg['lr']:.1e}")

## Training — Full Dataset

Train for exactly `config.epochs` epochs (the best epoch found in ablation).  
No validation loop; the scheduler is stepped on train loss.

In [ ]:
n_epochs = config.epochs
history  = {'train_loss': [], 'grad_norm': []}

print(f"Training for {n_epochs} epochs on {len(full_dataset)} users")
print("=" * 60)

for epoch in range(n_epochs):
    result = train_epoch(
        model, train_loader,
        config.loss, optimizer, device, config,
        clipper=clipper,
    )

    train_loss = result['loss']
    grad_norm  = result.get('grad_norm', float('nan'))
    current_lr = optimizer.param_groups[-1]['lr']

    scheduler.step(train_loss)

    history['train_loss'].append(train_loss)
    history['grad_norm'].append(grad_norm)

    print(
        f"Epoch {epoch+1:3d}/{n_epochs} "
        f"| Loss: {train_loss:.4f} "
        f"| Grad norm: {grad_norm:.3f} "
        f"| LR: {current_lr:.2e}"
    )

print("\nTraining complete!")

## Save Final Checkpoint

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir   = Path(CKPT_DIR) / f"{timestamp}_final_eval"
run_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        'model_state_dict': model.state_dict(),
        'config': cfg_dict,
        'history': history,
    },
    run_dir / 'final_checkpoint.pt',
)

print(f"Checkpoint saved to: {run_dir}")

## Test Prediction

We load the test file that includes gold labels (`test_labels_subtask1.csv`) through
the standard `EmoVADataset`, so the dataloader already carries both texts and gold values.
`collect_predictions_for_eval` runs the model and returns two dicts:
`predictions` and `gold`, both keyed by user_id with arrays of shape `[n_texts, 2]`.

In [ ]:
TEST_LABELS_PATH = (
    f"{PROJECT_ROOT}/dataset/TEST_LABELS_RELEASE_23FEB2026/test_labels_subtask1.csv"
)

test_dataset = EmoVADataset(
    path=TEST_LABELS_PATH,
    dtype=torch.float32,
    constrain_output=config.constrain_output,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,          # one user at a time — safe for any sequence length
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)

print(f"Test dataset: {len(test_dataset)} users")

predictions, gold = collect_predictions_for_eval(
    model, test_loader, device, verbose=True
)

print(f"Predictions collected for {len(predictions)} users")

## Official Evaluation

Metrics (from SemEval 2026 EmoVA Task 1 spec):
- **Between-user r** — Pearson r on per-user mean scores  
- **Within-user r** — mean of per-user Pearson r  
- **Composite r** — Fisher z-transform combination of the two (ranking metric)  
- **Overall composite r** — average of valence and arousal composite r

In [ ]:
results = evaluate_subtask1(predictions, gold, verbose=True)
print_evaluation_results(results, title="Test Set — Official Evaluation")

## Summary Table

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        'Dimension': 'Valence',
        'Between-user r': results['valence/r_between'],
        'Within-user r':  results['valence/r_within'],
        'Composite r':    results['valence/r_composite'],
        'Between MAE':    results['valence/mae_between'],
        'Within MAE':     results['valence/mae_within'],
    },
    {
        'Dimension': 'Arousal',
        'Between-user r': results['arousal/r_between'],
        'Within-user r':  results['arousal/r_within'],
        'Composite r':    results['arousal/r_composite'],
        'Between MAE':    results['arousal/mae_between'],
        'Within MAE':     results['arousal/mae_within'],
    },
]).set_index('Dimension').round(4)

print(summary.to_string())
print(f"\nOVERALL COMPOSITE r (ranking metric): {results['overall/r_composite']:.4f}")

## Scatter Plots — Predicted vs Gold (user means)

In [ ]:
import matplotlib.pyplot as plt

user_ids    = list(predictions.keys())
pred_v_mean = np.array([predictions[u][:, 0].mean() for u in user_ids])
pred_a_mean = np.array([predictions[u][:, 1].mean() for u in user_ids])
gold_v_mean = np.array([gold[u][:, 0].mean() for u in user_ids])
gold_a_mean = np.array([gold[u][:, 1].mean() for u in user_ids])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, gm, pm, dim, color, r_key in [
    (axes[0], gold_v_mean, pred_v_mean, 'Valence', 'steelblue',  'valence/r_between'),
    (axes[1], gold_a_mean, pred_a_mean, 'Arousal', 'seagreen',   'arousal/r_between'),
]:
    ax.scatter(gm, pm, alpha=0.55, s=20, c=color)
    lims = [min(gm.min(), pm.min()) - 0.1, max(gm.max(), pm.max()) + 0.1]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect')
    ax.set_xlabel(f'Gold {dim} (user mean)')
    ax.set_ylabel(f'Predicted {dim} (user mean)')
    ax.set_title(f"{dim} — Between-user r = {results[r_key]:.4f}")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(
    f"Overall Composite r = {results['overall/r_composite']:.4f}",
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(str(run_dir / 'test_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to {run_dir / 'test_scatter.png'}")

## Save Evaluation Results

In [ ]:
results_path = run_dir / 'eval_results.json'
with open(results_path, 'w') as f:
    json.dump({k: float(v) for k, v in results.items()}, f, indent=2)

print(f"Results saved to: {results_path}")